# 📝 Exercise: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า embedding, score, และ agent response ที่ต้องตรงกับรหัสนักศึกษา

## 📦 Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ !')

## 🎓 Enter student information

In [ ]:
# ─── ───
STUDENT_NAME = '' # ' '
STUDENT_ID = '' # '6512345678'
PHONE = '' # '081-234-5678'
LINE_ID = '' # 'somchai.j'

# ─── () ───
assert len(STUDENT_ID) >= 5, '❌ !'
assert len(STUDENT_NAME) >= 2, '❌ -!'

print(f'✅ -: {STUDENT_NAME}')
print(f'✅ : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Create a personalized dataset (do not edit this cell)

In [ ]:
%%time
# ===== Do not edit this cell =====
# Create a student-specific dataset from the student ID

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

---
## 🎯 1: Chunk + Embed + Search (3 )

- `homework_data/`
- Chunk `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- Embedding `intfloat/multilingual-e5-large`
- query: `MY_QUERY` ()
- Qdrant collection `f'hw_{STUDENT_ID}'`

**📝 :**
1. chunks?
2. Chunk similarity ? (score 4 )
3. Top-3 Qdrant score ?

In [ ]:
# 1: 

# 💡 Hint:
# 1. 'homework_data/' text 
# 2. from langchain_text_splitters import RecursiveCharacterTextSplitter
# 3. splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
# 4. chunks = splitter.split_text(all_text)
# 5. from sentence_transformers import SentenceTransformer
# 6. model = SentenceTransformer('intfloat/multilingual-e5-large')
# 7. passages = ['passage: ' + c for c in chunks]
# 8. embeddings = model.encode(passages)
# 9. query_emb = model.encode(f'query: {MY_QUERY}')
# 10. cosine_similarity() chunk 
# 11. from qdrant_client import QdrantClient, models
# 12. collection, upsert, query_points



# ✅ Self-check (uncomment )
# assert len(chunks) > 0, '❌ chunk'
# assert len(qdrant_results) == 3, '❌ top_k=3 Qdrant'
# print(f'✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

---
## 🎯 2: Agent + Custom Tool (3 )

- Gemini API Key (Colab Secrets)
- **Custom Tool** 1 ( BMI )
- **Agent** Google ADK Tool 
- → Agent Tool 

**📝 :**
1. Tool ? ( 1-2 )
2. output Agent Tool 
3. docstring ? ( 1-2 )

In [ ]:
# 2: 

# API Key
import os
try:
 from google.colab import userdata
 os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
 os.environ['GOOGLE_API_KEY'] = input('🔑 API Key: ')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# ─── Custom Tool ( BMI) ───
# 💡 : , GPA, 
def my_custom_tool(param1: float, param2: float) -> str:
 """ tool (! Agent docstring )

 Args:
 param1: parameter 1
 param2: parameter 2
 """
 # TODO: logic 
 result = 0 # 
 return f': {result}'

tool = FunctionTool(my_custom_tool)

# ─── Agent ───
my_agent = LlmAgent(
 name='my_assistant',
 model='gemini-2.5-flash',
 instruction='... ( tool )',
 tools=[tool]
)

# ─── ───
async def chat_with_agent(agent, message):
 runner = InMemoryRunner(agent=agent, app_name='homework')
 session = await runner.session_service.create_session(
 app_name='homework', user_id='student'
 )
 content = genai_types.Content(
 role='user', parts=[genai_types.Part(text=message)]
 )
 response_text = ''
 async for event in runner.run_async(
 user_id='student', session_id=session.id, new_message=content
 ):
 if event.content and event.content.parts:
 for part in event.content.parts:
 if part.text:
 response_text += part.text
 return response_text

# tool
answer = await chat_with_agent(my_agent, ' tool ')
print(f'🤖 Agent: {answer}')

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent '
print(f'✅ Step 2 passed!')

---
## 🎯 Step 3: RAG Agent + quality evaluation (4 points)

- Build a **RAG Tool** that searches Qdrant (using the collection from stepที่ 1)
- สร้าง **RAG Agent** ที่ใช้ RAG Tool ตอบคำถาม
- ถามคำถาม 3 ข้อ (กำหนดให้) → บันทึกคำตอบ
- ให้คะแนนคำตอบด้วย **LLM-as-Judge** (ใช้ Gemini ให้คะแนน 1-5)

**คำถามที่ต้องถาม:**
```python
questions = [
    f'query: {MY_QUERY}',   # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]
```

**📝 รายงาน:**
1. คำตอบของ RAG Agent ต่อ 3 คำถาม
2. LLM-as-Judge ให้คะแนนเท่าไร? (1-5 ต่อข้อ)
3. อธิบาย: Agent ตัดสินใจSearch from Qdrant อย่างไร? (2-3 ประโยค)

In [ ]:
# 3: 

# ─── A) RAG Tool ───
def search_knowledge(query: str) -> str:
 """ Qdrant
 AI, Machine Learning, NLP

 Args:
 query: 
 """
 # TODO: qdrant.query_points() collection Step 1
 # results = qdrant.query_points(collection_name=..., query=..., limit=3)
 # return string
 pass

# ─── B) RAG Agent ───
rag_agent = LlmAgent(
 name='rag_assistant',
 model='gemini-2.5-flash',
 instruction=' AI tool search_knowledge ',
 tools=[FunctionTool(search_knowledge)]
)

# ─── C) 3 ───
questions = [
 MY_QUERY, # query 
 'Embedding ?',
 ' RAG ?'
]

rag_answers = []
for q in questions:
 ans = await chat_with_agent(rag_agent, q)
 rag_answers.append({'question': q, 'answer': ans})
 print(f'\n{"="*50}')
 print(f'❓ {q}')
 print(f'🤖 {ans}')

# ─── D) LLM-as-Judge ───
from google import genai
judge_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

# 📋 Template judge ()
JUDGE_PROMPT = ''' AI

: {question}
: {answer}

 1-5 :
- 5 = 
- 4 = 
- 3 = 
- 2 = 
- 1 = 

 JSON: {{"score": 0, "reason": "..."}}
'''

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
for qa in rag_answers:
 prompt = JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer'])
 resp = judge_client.models.generate_content(
 model='gemini-2.5-flash', contents=prompt,
 config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
 )
 result = json.loads(resp.text)
 print(f' ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')

# ✅ Self-check
assert len(rag_answers) == 3, '❌ 3 '
print(f'\n✅ Step 3 passed: 3 + LLM-as-Judge !')

## 📊 

| | | |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline , Qdrant |
| 2. Agent + Custom Tool | 3 | Tool , Agent , docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent 3 , LLM-as-Judge , |
| **** | **10** | |

---
## ✅ Check answers

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [ ]:
# ===== cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 -: {STUDENT_NAME}')
print(f'🎓 : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 : _________________ ()')
print('=' * 50)
print()
print('📋 Checklist :')
print(' [ ] ')
print(' [ ] 1: Chunk + Embed + Qdrant ')
print(' [ ] 2: Agent + Tool ')
print(' [ ] 3: RAG Agent 3 + LLM-as-Judge')
print(' [ ] cell run ')